# Marked vs competing TTE sweep: architecture and sampling summary

This notebook is intentionally limited to the `marked` and `competing` DeepHit families.  It reads the point estimate and patient-bootstrap confidence interval already written to each complete official-result metric CSV; Seaborn is only used to draw the figures and is never asked to estimate an interval.

For repeated artifacts with the same family, sampling strategy, depth, kernel sizes, and random seed, the first result file in lexical path order is retained.


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(context='notebook', style='whitegrid')

# Complete raw official sweep.  Do not use final_model_selection/artifacts
# or the stale postprocessing metric_summary.csv here.
RESULT_DIR = 'results_official_experiment'

# One table is generated for every selection below.  Filters are matched
# exactly, so additional targets, metrics, or forecast horizons can be added
# without changing the downstream table code.
METRIC_SPECS = (
    {
        'key': 'macro_ph_end_0min',
        'label': 'Macro-pH AUROC at tracing end (0 min before delivery)',
        'filters': {
            'split': 'Test', 'target': 'macro_pH', 'metric': 'auroc_end_landmark',
            'minutes_before_delivery': 0.0,
        },
    },
    {
        'key': 'macro_ph_end_20min',
        'label': 'Macro-pH AUROC at 20 min before delivery',
        'filters': {
            'split': 'Test', 'target': 'macro_pH', 'metric': 'auroc_end_landmark',
            'minutes_before_delivery': 20.0,
        },
    },
    {
        'key': 'macro_ph_end_40min',
        'label': 'Macro-pH AUROC at 40 min before delivery',
        'filters': {
            'split': 'Test', 'target': 'macro_pH', 'metric': 'auroc_end_landmark',
            'minutes_before_delivery': 40.0,
        },
    },
    {
        'key': 'macro_ph_end_60min',
        'label': 'Macro-pH AUROC at 60 min before delivery',
        'filters': {
            'split': 'Test', 'target': 'macro_pH', 'metric': 'auroc_end_landmark',
            'minutes_before_delivery': 60.0,
        },
    },
)
METRIC_YLIM = (0.0, 1.0)
FAMILIES = ('marked', 'competing')
# Score each candidate by the mean of the four configured end-relative
# Macro-pH AUROCs. Each schema defines the grouping within which the
# highest-scoring candidate would be selected.
MODEL_SELECTION_SCHEMAS = {
    'family': ['family'],
    'family_and_sampling': ['family', 'sampling'],
    'family_and_architecture': ['family', 'depth', 'kernel_sizes'],
}

REPO_ROOT = Path.cwd().resolve()
RESULT_PATHS = [REPO_ROOT / RESULT_DIR]
if not RESULT_PATHS[0].is_dir():
    raise FileNotFoundError(f'Missing result directory: {RESULT_PATHS[0]}')
OUTPUT_DIR = REPO_ROOT / 'results_official_experiment' / 'postprocessing_sampling_architecture_sweep'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Reading raw sweep metrics from:')
for path in RESULT_PATHS:
    print(f'  {path}')
print(f'Writing figures and tables to {OUTPUT_DIR}')


In [ ]:
FAMILY_LABEL = {
    'marked': 'Marked DeepHit',
    'competing': 'Competing-risk DeepHit',
}
SAMPLING_LABEL = {
    'uniform': 'Uniform',
    'stratified_elapsed': 'Stratified elapsed',
    'terminal_balanced': 'Terminal balanced',
    'quartiles_plus_terminal': 'Quartiles + terminal',
}

# The official sweep uses 900-sample (one-hour at 0.25 Hz) input chunks.
# Each Inception block has three modules; following the widest-kernel path,
# its receptive field grows by 3 * (max_kernel - 1) samples per block.
INPUT_CHUNK_SAMPLES = 900
LATEX_MISSING_VALUE = '--'

def kernel_label(kernel_sizes: object) -> str:
    if not isinstance(kernel_sizes, (list, tuple)):
        return 'unknown'
    return '{' + ', '.join(map(str, sorted(kernel_sizes))) + '}'


def receptive_field_samples(depth: int, kernel_sizes: object) -> int:
    return 1 + 3 * int(depth) * (max(kernel_sizes) - 1)


def receptive_field_coverage_percent(receptive_field: int) -> int:
    return round(100 * receptive_field / INPUT_CHUNK_SAMPLES)


def receptive_field_label(receptive_field: int, coverage_percent: int) -> str:
    return f'RF {receptive_field} ({coverage_percent}%)'


def artifact_stem(metrics_path: Path) -> str:
    suffix = '_landmark_metrics.csv'
    if not metrics_path.name.endswith(suffix):
        raise ValueError(f'Unexpected metrics filename: {metrics_path.name}')
    return metrics_path.name.removesuffix(suffix)


def result_file_key(family: str, depth: object, kernels: object, sampling: str, metadata: dict, run_id: str) -> tuple[str, ...]:
    seed = metadata.get('random_state')
    if seed is None:
        match = re.search(r'(?:^|_)s(\d+)(?:_|$)', run_id)
        seed = match.group(1) if match else 'unknown'
    return (family, str(depth), json.dumps(kernels, sort_keys=True, default=str), str(sampling), str(seed))

def load_metric_rows() -> pd.DataFrame:
    frames = []
    seen_result_files = set()
    for metrics_path in sorted(
        metrics_path for result_path in RESULT_PATHS
        for metrics_path in result_path.glob('*_landmark_metrics.csv')
    ):
        manifest_path = metrics_path.with_name(f'{artifact_stem(metrics_path)}.json')
        if not manifest_path.is_file():
            print(f'Skipping {metrics_path.name}: matching manifest is missing.')
            continue
        try:
            metadata = json.loads(manifest_path.read_text())
        except (OSError, json.JSONDecodeError) as error:
            print(f'Skipping {metrics_path.name}: unreadable manifest ({error}).')
            continue
        family = metadata.get('mode')
        if family not in FAMILIES:
            continue
        fit_kwargs = (metadata.get('model') or {}).get('fit_kwargs') or {}
        depth = fit_kwargs.get('depth')
        kernels = fit_kwargs.get('kernel_sizes')
        sampling = metadata.get('training_sampling_strategy')
        if depth is None or not kernels or not sampling:
            print(f'Skipping {metrics_path.name}: depth, kernel sizes, or sampling metadata is unavailable.')
            continue
        run_id = artifact_stem(metrics_path)
        selection_key = result_file_key(family, depth, kernels, sampling, metadata, run_id)
        if selection_key in seen_result_files:
            print(f'Skipping duplicate result file (keeping first): {metrics_path.name}')
            continue
        seen_result_files.add(selection_key)
        try:
            metrics = pd.read_csv(metrics_path)
        except (OSError, pd.errors.EmptyDataError, pd.errors.ParserError) as error:
            print(f'Skipping {metrics_path.name}: unreadable metrics ({error}).')
            continue
        for spec in METRIC_SPECS:
            frame = metrics
            for column, expected in spec['filters'].items():
                if column not in frame:
                    frame = frame.iloc[0:0]
                    break
                frame = frame.loc[frame[column].isna()] if pd.isna(expected) else frame.loc[frame[column].eq(expected)]
            if frame.empty:
                print(f"Skipping {metrics_path.name} ({spec['key']}): no matching metric row.")
                continue
            if len(frame) > 1:
                print(f"{metrics_path.name} ({spec['key']}): found {len(frame)} matching metric rows; keeping the first.")
            frame = frame.iloc[[0]].copy()
            frame['metric_key'] = spec['key']
            frame['metric_label'] = spec['label']
            frame['family'] = family
            frame['family_label'] = FAMILY_LABEL[family]
            frame['sampling'] = sampling
            frame['sampling_label'] = SAMPLING_LABEL.get(sampling, sampling.replace('_', ' ').title())
            frame['depth'] = depth
            frame['kernel_sizes'] = kernel_label(kernels)
            receptive_field = receptive_field_samples(depth, kernels)
            coverage_percent = receptive_field_coverage_percent(receptive_field)
            frame['receptive_field_samples'] = receptive_field
            frame['receptive_field_coverage_percent'] = coverage_percent
            frame['receptive_field'] = receptive_field_label(receptive_field, coverage_percent)
            frame['setting_label'] = (
                'depth=' + frame['depth'].astype(str)
                + ', kernels=' + frame['kernel_sizes'] + '; ' + frame['receptive_field']
            )
            frame['run_id'] = run_id
            frame['n_artifacts'] = 1
            frame['source_file'] = metrics_path.name
            frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


metric_rows = load_metric_rows()
assert not metric_rows.empty, (
    'No marked/competing raw official-result files matched METRIC_SPECS.'
)
metric_rows[['value', 'ci_lower', 'ci_upper']] = metric_rows[['value', 'ci_lower', 'ci_upper']].apply(pd.to_numeric, errors='coerce')
metric_rows = metric_rows.loc[metric_rows['value'].notna()].copy()
invalid_ci = metric_rows.loc[
    metric_rows['ci_lower'].notna()
    & metric_rows['ci_upper'].notna()
    & ~metric_rows['ci_lower'].le(metric_rows['value'])
    | metric_rows['ci_upper'].notna()
    & ~metric_rows['value'].le(metric_rows['ci_upper'])
]
assert invalid_ci.empty, 'Some reported confidence intervals do not contain their point estimate.'
display(metric_rows[['metric_label', 'family_label', 'sampling_label', 'depth', 'kernel_sizes', 'receptive_field', 'value', 'ci_lower', 'ci_upper', 'source_file']])


In [ ]:
# This is an aggregation of values already computed by the evaluator.  In
# particular, no bootstrap resampling and no Seaborn errorbar calculation is
# performed here.
GROUP_COLUMNS = [
    'metric_key', 'metric_label', 'family', 'family_label', 'sampling', 'sampling_label', 'depth', 'kernel_sizes',
    'receptive_field_samples', 'receptive_field_coverage_percent', 'setting_label',
]
summary = metric_rows.groupby(GROUP_COLUMNS, dropna=False, as_index=False).agg(
    mean=('value', 'mean'),
    ci_lower=('ci_lower', 'mean'),
    ci_upper=('ci_upper', 'mean'),
    n_artifacts=('n_artifacts', 'sum'),
)
summary['ci_text'] = np.where(
    summary['ci_lower'].notna() & summary['ci_upper'].notna(),
    summary.apply(lambda row: f"{row['mean']:.3f}\n[{row['ci_lower']:.3f}, {row['ci_upper']:.3f}]", axis=1),
    summary['mean'].map(lambda value: f'{value:.3f}\n[CI unavailable]'),
)

# The heatmap keeps the strategy with the largest mean metric at each
# architecture.  Ties are resolved alphabetically for reproducibility.
architecture_columns = ['metric_key', 'metric_label', 'family', 'family_label', 'depth', 'kernel_sizes']
heatmap_summary = (
    summary.sort_values(architecture_columns + ['mean', 'sampling_label'],
                        ascending=[True] * len(architecture_columns) + [False, True])
    .groupby(architecture_columns, as_index=False, sort=False)
    .first()
)
heatmap_summary['ci_text'] = heatmap_summary.apply(
    lambda row: f"{row['mean']:.3f}\n[{row['ci_lower']:.3f}, {row['ci_upper']:.3f}]\n{row['sampling_label']}"
    if pd.notna(row['ci_lower']) and pd.notna(row['ci_upper'])
    else f"{row['mean']:.3f}\n[CI unavailable]\n{row['sampling_label']}",
    axis=1,
)
display(summary.drop(columns='n_artifacts').sort_values(
    ['metric_label', 'family_label', 'sampling_label', 'receptive_field_coverage_percent', 'receptive_field_samples']
))
display(heatmap_summary.sort_values(['metric_label', 'family_label', 'depth', 'kernel_sizes']))


In [ ]:
# One heatmap per model family.  Rows are kernel-size lists and columns are
# depths; each tile is the best sampling strategy for that architecture.
family_order = [family for family in FAMILIES if family in set(heatmap_summary['family'])]
for spec in METRIC_SPECS:
    metric_data = heatmap_summary.loc[heatmap_summary['metric_key'].eq(spec['key'])]
    for family in family_order:
        data = metric_data.loc[metric_data['family'].eq(family)].copy()
        if data.empty:
            continue
        depths = sorted(data['depth'].unique())
        kernels = list(dict.fromkeys(data.sort_values('kernel_sizes')['kernel_sizes']))
        values = data.pivot(index='kernel_sizes', columns='depth', values='mean').reindex(index=kernels, columns=depths)
        annotations = data.pivot(index='kernel_sizes', columns='depth', values='ci_text').reindex(index=kernels, columns=depths).fillna('')
        # FacetGrid supplies a fresh Seaborn-owned figure for every heatmap.
        grid = sns.FacetGrid(pd.DataFrame({'family': [family]}), height=7.0, aspect=1.45, despine=False)
        chart = sns.heatmap(
            values, annot=annotations, fmt='', cmap='crest', vmin=METRIC_YLIM[0], vmax=METRIC_YLIM[1],
            linewidths=0.7, linecolor='white', cbar_kws={'label': spec['label']}, ax=grid.axes.flat[0],
        )
        chart.set(
            title=f"{FAMILY_LABEL[family]}: best sampling strategy per architecture",
            xlabel='Inception depth', ylabel='Kernel sizes',
        )
        grid.figure.tight_layout()
        grid.figure.savefig(
            OUTPUT_DIR / f"{spec['key']}_{family}_best_sampling_architecture_heatmap.png",
            dpi=220, bbox_inches='tight',
        )
        grid.figure.show()


In [ ]:
# Sampling-strategy tables.  Each table corresponds to one model-family row
# from the former barplot grid, with sampling strategies across columns and
# architecture settings down rows.  Cells retain the reported point estimate
# and bootstrap CI; the setting label includes receptive-field coverage.
def format_metric_cell(row: pd.Series) -> str:
    if pd.isna(row['mean']):
        return LATEX_MISSING_VALUE
    if pd.notna(row['ci_lower']) and pd.notna(row['ci_upper']):
        interval = f" [{row['ci_lower']:.3f}, {row['ci_upper']:.3f}]"
    else:
        interval = ' [CI unavailable]'
    return f"{row['mean']:.3f}{interval}"

# Preserve a machine-readable version of all displayed values alongside the
# family-specific presentation tables.
table_data = summary.sort_values([
    'metric_key', 'family_label', 'receptive_field_coverage_percent', 'receptive_field_samples',
    'sampling_label',
]).copy()
table_data.to_csv(OUTPUT_DIR / 'sampling_strategy_by_architecture_table_data.csv', index=False)

for spec in METRIC_SPECS:
    metric_summary = summary.loc[summary['metric_key'].eq(spec['key'])].copy()
    if metric_summary.empty:
        print(f"No completed results for {spec['label']}.")
        continue
    setting_order = list(dict.fromkeys(metric_summary.sort_values(
        ['receptive_field_coverage_percent', 'receptive_field_samples', 'depth', 'kernel_sizes']
    )['setting_label']))
    sampling_order = list(dict.fromkeys(metric_summary['sampling_label']))
    for family in family_order:
        family_data = metric_summary.loc[metric_summary['family'].eq(family)].copy()
        if family_data.empty:
            continue
        family_data['result'] = family_data.apply(format_metric_cell, axis=1)
        scores = (
            family_data.pivot(index='setting_label', columns='sampling_label', values='mean')
            .reindex(index=setting_order, columns=sampling_order)
        )
        table = (
            family_data.pivot(index='setting_label', columns='sampling_label', values='result')
            .reindex(index=setting_order, columns=sampling_order)
            .fillna(LATEX_MISSING_VALUE)
        )
        table.index.name = 'Architecture setting'
        table.columns.name = 'Sampling strategy'

        def bold_row_best(row: pd.Series) -> list[str]:
            row_scores = scores.loc[row.name]
            best_score = row_scores.max()
            return [
                'font-weight: bold'
                if pd.notna(score) and np.isclose(score, best_score)
                else ''
                for score in row_scores
            ]

        # Escape %, _, braces, and other TeX-sensitive content in labels and
        # values.  `convert_css=True` below turns the displayed bold cells
        # into TeX bold commands in the exported tabular.
        styled_table = table.style.apply(bold_row_best, axis=1).format(escape='latex')
        output_stem = f"{spec['key']}_{family}_sampling_strategy_by_architecture_table"
        print(f"\n{FAMILY_LABEL[family]}: {spec['label']}")
        display(styled_table)
        table.to_csv(OUTPUT_DIR / f'{output_stem}.csv')
        styled_table.to_latex(
            OUTPUT_DIR / f'{output_stem}.tex',
            column_format='l' + 'c' * len(table.columns),
            hrules=True,
            convert_css=True,
        )


In [ ]:
# Model-selection score tables. Each candidate score is the mean of the
# end-relative Macro-pH AUROCs selected in METRIC_SPECS. The schema changes
# only the group in which candidates compete; it does not change a score.
SELECTION_CANDIDATE_COLUMNS = [
    'family', 'family_label', 'sampling', 'sampling_label', 'depth',
    'kernel_sizes', 'receptive_field_samples',
    'receptive_field_coverage_percent', 'receptive_field', 'setting_label',
]

def selection_scores_for_schema(schema_name: str, group_columns: list[str]) -> pd.DataFrame:
    scores = metric_rows.groupby(SELECTION_CANDIDATE_COLUMNS, dropna=False, as_index=False).agg(
        selection_score=('value', 'mean'),
        selection_metric_rows=('metric_key', 'nunique'),
    )
    scores = scores.loc[scores['selection_metric_rows'].eq(len(METRIC_SPECS))].copy()
    if scores.empty:
        return scores
    scores['selection_schema'] = schema_name
    scores['selection_group'] = scores[group_columns].fillna('Missing').astype(str).agg(' — '.join, axis=1)
    scores = scores.sort_values(
        ['selection_group', 'selection_score', 'setting_label', 'sampling_label'],
        ascending=[True, False, True, True],
    )
    scores['selection_rank'] = scores.groupby('selection_group', sort=False).cumcount() + 1
    scores['selected'] = scores['selection_rank'].eq(1)
    return scores

selection_score_frames = [
    selection_scores_for_schema(schema_name, group_columns)
    for schema_name, group_columns in MODEL_SELECTION_SCHEMAS.items()
]
selection_scores = pd.concat(
    [frame for frame in selection_score_frames if not frame.empty],
    ignore_index=True,
) if any(not frame.empty for frame in selection_score_frames) else pd.DataFrame()
if selection_scores.empty:
    print('No complete candidates are available for model-selection scoring.')
else:
    selection_columns = [
        'selection_schema', 'selection_group', 'family_label', 'sampling_label',
        'depth', 'kernel_sizes', 'receptive_field', 'selection_score',
        'selection_metric_rows', 'selection_rank', 'selected', 'source_file',
    ]
    source_files = metric_rows.groupby(SELECTION_CANDIDATE_COLUMNS, dropna=False)['source_file'].agg(
        lambda values: '; '.join(sorted(set(values)))).rename('source_file').reset_index()
    selection_scores = selection_scores.merge(source_files, on=SELECTION_CANDIDATE_COLUMNS, how='left')
    selection_scores = selection_scores.sort_values(
        ['selection_schema', 'selection_group', 'selection_rank']
    ).reset_index(drop=True)
    selection_scores.to_csv(OUTPUT_DIR / 'model_selection_schema_scores.csv', index=False)
    winners = selection_scores.loc[selection_scores['selected'], selection_columns].copy()
    winners.to_csv(OUTPUT_DIR / 'model_selection_schema_winners.csv', index=False)
    winners.to_latex(
        OUTPUT_DIR / 'model_selection_schema_winners.tex', index=False, escape=True,
        float_format=lambda value: f'{value:.3f}',
    )
    display(selection_scores[selection_columns])
    print(f'Wrote {OUTPUT_DIR / "model_selection_schema_scores.csv"}')
    print(f'Wrote {OUTPUT_DIR / "model_selection_schema_winners.csv"}')
